# Vnstock — Vietnam market data for prediction

Use the same **Python kernel** as your project (`.venv`).

- **Live quotes**: `stock.trading.price_board(...)` (intraday board; not used as model input).
- **Historical OHLCV**: `fetch_vnstock` in `src/data/loaders.py` wraps `stock.quote.history(...)` and returns the same columns as the Nasdaq path expects for `add_all_features` (`open`, `high`, `low`, `close`, `volume`; no `adj_close` — see `compute_target_price` in `preprocess.py`).

With **vnstock 3.5.x**, `source="VCI"` often fails during client init; **`KBS`** is the reliable default for daily history in this environment.

In [1]:
import sys
from pathlib import Path

# Project root (parent of notebooks/)
ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import vnstock

📦 **Vnstock 4.0.1 is available**
Current: 3.5.1 (Python 3.13 (venv))
Update: `/Users/cps/DL4AI-240166-project-1/.venv/bin/python -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.8 is available**
Current: 2.4.6 (Python 3.13 (venv))
Update: `/Users/cps/DL4AI-240166-project-1/.venv/bin/python -m pip install vnai --upgrade`
Release: https://pypi.org/project/vnai/#history

In [2]:
# vnstock 3.x may not expose __version__ on the module — use metadata instead
from importlib.metadata import version, PackageNotFoundError

try:
    print("vnstock version:", version("vnstock"))
except PackageNotFoundError:
    print("vnstock not installed in this kernel — pick .venv interpreter")

vnstock version: 3.5.1


In [3]:
from vnstock import Vnstock

stock = Vnstock().stock(symbol='FPT', source='KBS')

# gọi trading qua object
df_board = stock.trading.price_board(['FPT'])

print(df_board.head())

  symbol           time exchange  ceiling_price  floor_price  reference_price  \
0    FPT  1777358503278     HOSE          78500        68300            73400   

   open_price  high_price  low_price  close_price  ...  bid_vol_3  \
0       73400       74400      73000        73200  ...     128700   

   ask_price_1  ask_vol_1  ask_price_2  ask_vol_2 ask_price_3  ask_vol_3  \
0      73200.0      15000        73300      58000       73400      67500   

   foreign_buy_volume  foreign_sell_volume  foreign_room  
0              579826              3362540     295444091  

[1 rows x 30 columns]


In [4]:
import time

import pandas as pd

from src.data.loaders import fetch_vnstock

OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HOSE names with decent liquidity — edit freely
TICKERS = ["FPT", "VCB", "VHM", "VNM", "HPG"]
START = "2020-01-01"
END = pd.Timestamp.today().strftime("%Y-%m-%d")
SOURCE = "KBS"

frames = []
for sym in TICKERS:
    try:
        d = fetch_vnstock(sym, START, END, source=SOURCE)
        d = d.assign(ticker=sym)
        frames.append(d)
        single = OUT_DIR / f"{sym}_ohlcv.csv"
        d.to_csv(single)
        print(f"{sym}: {len(d)} rows → {single}")
    except Exception as exc:
        print(f"{sym}: ERROR {type(exc).__name__}: {exc}")
    time.sleep(0.35)

if frames:
    panel = pd.concat(frames, axis=0).reset_index().sort_values(["date", "ticker"])
    combined = OUT_DIR / "vietnam_ohlcv_panel.csv"
    panel.to_csv(combined, index=False)
    print(f"Combined panel: {combined}  shape={panel.shape}")
else:
    print("No frames — check network / ticker symbols / vnstock source.")

FPT: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/FPT_ohlcv.csv
VCB: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VCB_ohlcv.csv
VHM: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VHM_ohlcv.csv
VNM: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VNM_ohlcv.csv
HPG: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/HPG_ohlcv.csv
Combined panel: /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/vietnam_ohlcv_panel.csv  shape=(7875, 7)


In [35]:
import time

import pandas as pd

from src.data.loaders import fetch_vnstock

OUT_DIR = ROOT / "notebooks" / "data" / "vietnam" / "csv"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# HOSE names with decent liquidity — edit freely
TICKERS = {
    'VIC',
    'TCB',
    'MSN',
    'MWG',
    'VND'
}
START = "2020-01-01"
END = pd.Timestamp.today().strftime("%Y-%m-%d")
SOURCE = "KBS"

frames = []
for sym in TICKERS:
    try:
        d = fetch_vnstock(sym, START, END, source=SOURCE)
        d = d.assign(ticker=sym)
        frames.append(d)
        single = OUT_DIR / f"{sym}_ohlcv.csv"
        d.to_csv(single)
        print(f"{sym}: {len(d)} rows → {single}")
    except Exception as exc:
        print(f"{sym}: ERROR {type(exc).__name__}: {exc}")
    time.sleep(0.35)

if frames:
    panel = pd.concat(frames, axis=0).reset_index().sort_values(["date", "ticker"])
    combined = OUT_DIR / "vietnam_ohlcv_panel.csv"
    panel.to_csv(combined, index=False)
    print(f"Combined panel: {combined}  shape={panel.shape}")
else:
    print("No frames — check network / ticker symbols / vnstock source.")

MSN: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/MSN_ohlcv.csv
TCB: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/TCB_ohlcv.csv
VIC: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VIC_ohlcv.csv
VND: 1569 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/VND_ohlcv.csv
MWG: 1575 rows → /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/MWG_ohlcv.csv
Combined panel: /Users/cps/DL4AI-240166-project-1/notebooks/data/vietnam/csv/vietnam_ohlcv_panel.csv  shape=(7869, 7)
